# 🧬 Benchmarking Lightweight Mamba Models for Polyp Segmentation (1-Seed Baseline)
> **Quy trình hoạt động:**
> 1. Cài đặt môi trường & thư viện chuẩn.
> 2. **Bạn điền lệnh cài đặt file wheel `mamba_ssm` compiled vào Cell 2.**
> 3. Bấm **Runtime -> Run all** (hoặc chạy từ đầu đến cuối).
> 4. Hệ thống sẽ **tự động huấn luyện lần lượt từng mô hình**, đánh giá Benchmark, tạo ảnh trực quan hóa, và **tự động đóng gói thành từng file zip độc lập cho từng mô hình rồi tải về máy**.
> 5. Mỗi mô hình chỉ chạy 1 seed (`--seed 42`) với loss chuẩn (`structure_loss`) để lấy kết quả đối chiếu baseline ban đầu, tiết kiệm tài nguyên GPU tối đa cho bước cải tiến tiếp theo.

In [ ]:
# ==============================================================================
# 1. CLONE REPOSITORY & CÀI ĐẶT THƯ VIỆN CƠ BẢN
# ==============================================================================
!git clone https://github.com/doantrongthai/Polyp_Segmentation_Research.git
%cd Polyp_Segmentation_Research
!pip install -q -r requirements.txt einops timm monai

### ⚙️ CELL DÀNH RIÊNG CHO BẠN: CÀI ĐẶT MAMBA ĐÃ COMPILE
> **Hãy dán lệnh tải và cài đặt file wheel (`.whl`) của bạn vào cell dưới đây trước khi chạy toàn bộ notebook:**

In [ ]:
# ==============================================================================
# [BỎ TRỐNG ĐỂ BẠN ĐIỀN LỆNH CÀI MAMBA RIÊNG VÀO ĐÂY]
# Ví dụ:
# !pip install /path/to/causal_conv1d-*.whl
# !pip install /path/to/mamba_ssm-*.whl
# hoặc tải từ Google Drive / Link trực tiếp:
# !gdown <id_causal_conv1d> && pip install causal_conv1d-*.whl
# !gdown <id_mamba_ssm> && pip install mamba_ssm-*.whl
# ==============================================================================



In [ ]:
# ==============================================================================
# 2. KIỂM TRA MÔI TRƯỜNG CUDA & MAMBA CHÍNH THỨC
# ==============================================================================
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

try:
    import mamba_ssm
    from mamba_ssm import Mamba
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print(f"✅ mamba_ssm phiên bản {getattr(mamba_ssm, '__version__', 'CUDA')} đã sẵn sàng hoạt động!")
except Exception as e:
    print(f"⚠️ Cảnh báo Mamba: {e}")
    print("Lưu ý: Nếu bạn chưa chạy lệnh cài mamba ở Cell trên, vui lòng chạy lệnh cài trước khi train.")

from models import list_models
print("\nDanh sách các mô hình trong Registry:")
for m in sorted(list_models()):
    if any(k in m for k in ['mamba', 'vmunet', 'bampolyp']):
        print(f"  [Mamba Target] -> {m}")

---
## 🚀 CHẠY TỰ ĐỘNG TUẦN TỰ TỪNG MÔ HÌNH MAMBA & TẢI TỪNG FILE ZIP RIÊNG BIỆT
> **Quy trình chuẩn cho mỗi mô hình:**
> 1. Làm sạch thư mục tạm `snapshots/`, `results/`, `visualizations/`, `logs/` để đảm bảo kết quả không bị trộn lẫn.
> 2. Chạy `train.py` với cấu hình chuẩn: `--model <name> --loss structure_loss --seed 42` (tự động đánh giá đầy đủ 5 tập test và đo đạc Edge AI: Params, FLOPs, Latency, FPS).
> 3. Chạy `visualize.py` sinh ảnh so sánh viền polyp thực tế.
> 4. Đóng gói thành `PolypSeg_<model_name>_seed42.zip`.
> 5. Gọi lệnh tải trực tiếp về máy tính cá nhân ngay sau khi model đó hoàn tất.

In [ ]:
# ==============================================================================
# 3. PIPELINE CHẠY TỪNG MODEL VÀ TẢI ZIP RIÊNG TỪNG CÁI
# ==============================================================================
import os
import shutil
from google.colab import files

# Danh sách 4 mô hình Mamba nhẹ cần chạy baseline
mamba_models = [
    'ultralight_vmunet',  # 0.049M params (UltraLight VM-UNet)
    'lightm_unet',        # 1.34M params  (LightM-UNet 2D)
    'bampolyp',           # 6.66M params  (BAMPolyp ICCV 2025)
    'h_vmunet'            # 8.97M params  (H-vmunet)
]

completed_packages = []

for idx, model_name in enumerate(mamba_models, start=1):
    print("\n" + "=" * 70)
    print(f"[{idx}/{len(mamba_models)}] BẮT ĐẦU: MODEL '{model_name.upper()}' (SEED 42)")
    print("=" * 70 + "\n")
    
    # Bước A: Dọn dẹp sạch sẽ thư mục kết quả cũ để zip hoàn toàn riêng biệt
    for folder in ['snapshots', 'results', 'visualizations', 'logs']:
        if os.path.exists(folder):
            shutil.rmtree(folder)
            
    # Bước B: Huấn luyện mô hình (1 seed 42, 100 epochs, structure_loss)
    # Script tự động thực hiện Final Benchmark (Params, FLOPs, Latency, FPS, 5 Datasets)
    exit_code = os.system(f"python train.py --model {model_name} --loss structure_loss --seed 42")
    if exit_code != 0:
        print(f"❌ Huấn luyện '{model_name}' bị gián đoạn (Exit code: {exit_code}). Bỏ qua model này.")
        continue
        
    # Bước C: Sinh ảnh trực quan hóa định tính
    weights_path = f"snapshots/{model_name}/best.pth"
    if os.path.exists(weights_path):
        os.system(f"python visualize.py --model {model_name} --weights {weights_path} --dataset all --num_samples 5")
    else:
        print(f"⚠️ Không tìm thấy checkpoint best.pth tại: {weights_path}")
        
    # Bước D: Nén kết quả thành 1 file zip riêng của model này
    zip_filename = f"/content/PolypSeg_{model_name}_seed42.zip"
    if os.path.exists(zip_filename):
        os.remove(zip_filename)
        
    print(f"\n[Nén Package] Đang nén kết quả cho '{model_name}'...")
    os.system(f"zip -q -r {zip_filename} snapshots/ results/ visualizations/ logs/")
    
    if os.path.exists(zip_filename):
        file_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
        print(f"✅ Đã tạo thành công: {zip_filename} ({file_size_mb:.2f} MB)")
        completed_packages.append(zip_filename)
        
        # Bước E: Tự động tải file zip về máy cá nhân
        try:
            print(f"[Download] Đang gửi lệnh tải xuống trình duyệt: {os.path.basename(zip_filename)}...")
            files.download(zip_filename)
        except Exception as err:
            print(f"⚠️ Không thể popup download tự động ({err}). File zip vẫn được giữ nguyên tại: {zip_filename}")
    else:
        print(f"❌ Lỗi khi nén file zip cho model {model_name}")

print("\n" + "=" * 70)
print("🎉 HOÀN TẤT HUẤN LUYỆN TOÀN BỘ CÁC MÔ HÌNH MAMBA!")
print("Danh sách file zip đã tạo:")
for pkg in completed_packages:
    print(f"  - {pkg} ({os.path.getsize(pkg)/(1024*1024):.2f} MB)")
print("=" * 70)